In [ ]:
# Cell 1 — Install + clone + set API key
# Set your API key in Kaggle Secrets BEFORE running:
#   Kaggle sidebar → Add-ons → Secrets → Add secret
#   Name: GROQ_API_KEY   (or OPENAI_API_KEY or ANTHROPIC_API_KEY)

!pip install transformers scikit-learn datasets groq openai anthropic -q

import os, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# ── Load API key from Kaggle Secrets ──────────────────────────────────────────
secrets = UserSecretsClient()

PROVIDER = 'groq'   # <-- change to 'openai' or 'anthropic' if needed

if PROVIDER == 'groq':
    os.environ['GROQ_API_KEY'] = secrets.get_secret('GROQ_API_KEY')
    print('Groq key loaded')
elif PROVIDER == 'openai':
    os.environ['OPENAI_API_KEY'] = secrets.get_secret('OPENAI_API_KEY')
    print('OpenAI key loaded')
elif PROVIDER == 'anthropic':
    os.environ['ANTHROPIC_API_KEY'] = secrets.get_secret('ANTHROPIC_API_KEY')
    print('Anthropic key loaded')

# ── Clone / update repo ───────────────────────────────────────────────────────
REPO = Path('/kaggle/working/multihop-retrieval')
if REPO.exists():
    os.system(f'git -C {REPO} pull')
else:
    os.system(f'git clone https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git {REPO}')

sys.path.insert(0, str(REPO / 'delta_system'))
print('Ready.')

In [ ]:
# Cell 2 — Run LLM vs bert_maxsim comparison
# Evaluates on 200 IteraTeR meaning-changed pairs.
# No training needed — uses existing wiki_model.pt checkpoint.

import subprocess, sys

PROVIDER = 'groq'   # match Cell 1
N        = 200      # pairs to evaluate (each = 1 API call)

cmd = [
    sys.executable,
    str(Path('/kaggle/working/multihop-retrieval/delta_system/gpt_comparison_eval.py')),
    '--provider', PROVIDER,
    '--n',        str(N),
    '--ckpt',     '/kaggle/working/checkpoints/wiki_model.pt',
    '--delay',    '0.2',   # 0.2s between calls; increase to 1.0 if rate-limited
]

result = subprocess.run(cmd, capture_output=False)
if result.returncode != 0:
    print('Process exited with error code', result.returncode)